In [0]:
from pyspark.sql.functions import col, count, when, isnan, avg, sum as spark_sum, lit, desc
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import pandas as pd

# Adding to avoid casting errors when converting weather string columns to numeric
spark.conf.set("spark.sql.ansi.enabled", "false")

In [0]:
# df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")

# # Checkpoint as parquet (run once, then comment out)
# TEAM_DIR = "dbfs:/student-groups/Group_3_2"
# dbutils.fs.mkdirs(TEAM_DIR)
# df_otpw.write.mode("overwrite").parquet(f"{TEAM_DIR}/otpw_3m.parquet")
# print("Parquet saved.")

In [0]:
# Always read from parquet
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
#df = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")
df = spark.read.parquet(f"{TEAM_DIR}/otpw_12m.parquet")

# Phase 2 Feature engineering

In [0]:
# Filter out cancelled flights
df = df.filter(col("CANCELLED") == 0)
num_rows = df.count()
print(f"After removing cancelled flights: {num_rows:,} rows remaining")

In [0]:
# Filter FM-15
df = df.filter(col("REPORT_TYPE") == "FM-15")
num_rows = df.count()
print(f"After filtering FM-15: {num_rows:,} rows remaining")

In [0]:
df = df.withColumn(
    "PrecipBinary",
    when(col("HourlyPrecipitation") > 0, 1).otherwise(0)
)

In [0]:
from pyspark.sql.functions import mean, stddev

for c in {"HourlyRelativeHumidity", "HourlyDryBulbTemperature"}:
    stats = df.select(mean(c).alias("mean"), stddev(c).alias("std")).collect()[0]
    df = df.withColumn(c + "_scaled", (col(c) - stats["mean"]) / stats["std"])

In [0]:
from pyspark.sql.functions import log1p

# Log-transform (creates a new column)
df = df.withColumn("HourlyWindSpeed_log", log1p(col("HourlyWindSpeed")))

# Compute mean and std
stats = df.select(
    mean("HourlyWindSpeed_log").alias("mean"),
    stddev("HourlyWindSpeed_log").alias("std")
).collect()[0]

# Scale to mean=0, std=1
df = df.withColumn(
    "HourlyWindSpeed_scaled",
    (col("HourlyWindSpeed_log") - stats["mean"]) / stats["std"]
)

In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "VisibilityCat",
    when(col("HourlyVisibility") < 2, 0)
    .when((col("HourlyVisibility") >= 2) & (col("HourlyVisibility") < 5), 1)
    .when((col("HourlyVisibility") >= 5) & (col("HourlyVisibility") < 10), 2)
    .otherwise(3)
)

In [0]:
df = df.withColumn(
    "IsWeekend",
    when((col("DAY_OF_WEEK") == 6) | (col("DAY_OF_WEEK") == 7), 1).otherwise(0)
)

In [0]:
from pyspark.sql.functions import concat_ws, col

df = df.withColumn(
    "Route",
    concat_ws("_", col("ORIGIN"), col("DEST"))
)

In [0]:
display(df)

#Phase 3 Feature engineering

We create a window 2-4 hours before the flight to see the count percentange of delayed flights. If there are no flights in that window, it is it counts as 0 (nulls are replaced with 0).

We can change the flight window to capture more time before

The windows are grouped by airport (not enough data for it to be grouped by airline in my opinion).

In [0]:
from pyspark.sql import functions as F

# string to timestamp
df = df.withColumn(
    "sched_ts",
    F.to_timestamp("sched_depart_date_time")
)


In [0]:
from pyspark.sql.window import Window

# for performance
df = df.repartition("ORIGIN")

window_spec = Window.partitionBy("ORIGIN") \
    .orderBy(F.col("sched_ts").cast("long")) \
    .rangeBetween(-4 * 3600, -2 * 3600) # can change these number for different window

In [0]:
# count of delayed fights
df = df.withColumn(
    "delayed_2_4hr_before",
    F.coalesce(
        F.sum("DEP_DEL15").over(window_spec),
        F.lit(0) # replace null with 0
    )
)

In [0]:
# percent of delayed fights
df = df.withColumn(
    "delay_pct_2_4hr",
    F.coalesce(
        F.avg(F.coalesce(F.col("DEP_DEL15"), F.lit(0))).over(window_spec),
        F.lit(0) # replace null with 0
    )
)

In [0]:
display(df)

Could be leakage concerns since we are using scheduled depart time instead of actual depart time. However, we would know if a flight would be delayed if it has not departed by its actual flight time. Only the 15 minute window we are concerned about. If necessary, we can do 2h15m - 4h15m before.